# Create sessions.csv: which data to use (to discuss)



*Notebook variables (next cell)*

| Variable  | Meaning |
|---|---|
| `MIN_LEN` | sessions with fewer listens are dropped |
| `MAX_LEN` | longer sessions keep only their first 25 listens (the rest is dropped, not made into a new session) |
| `MIN_GAP` | a gap longer than this, from the start of one listen to the start of the next, starts a new session |

* And SAMPLE_PATH & SESSION_PATH

In [1]:
# define variables
import pandas as pd
from pathlib import Path

MIN_LEN = 5                         # minimum number of listens (songs) in a session
MAX_LEN = 25                        # maximum number of listens (songs) in a sessions 
MIN_GAP = pd.Timedelta(minutes=30)  # a gap longer than this starts a new session (start of A to start of B)

SAMPLE_PATH = Path("../../Samples/sample_2023_600users.csv") # input
SESSIONS_PATH = Path("sessions.csv") # output

Clean data 
* remove duplicate listens: same user id, track id, timestamp
* get genre data if not there already


In [2]:
# Clean data: read the sample and remove duplicate listens (same user, track, timestamp)
df = pd.read_csv(SAMPLE_PATH)
df["listen_time_utc"] = pd.to_datetime(df["listen_time_utc"], utc=True)

# track id: matchkey (97% filled); fall back to lowercased artist + song when it is missing
df["track_key"] = df["matchkey"].fillna(df["artist"].str.lower() + " - " + df["song"].str.lower())

n_before = len(df)
df = df.drop_duplicates(subset=["user_id", "track_key", "listen_time_utc"])
print(f"Removed {n_before - len(df):,} duplicate listens, {len(df):,} left")

Removed 12 duplicate listens, 199,995 left


Split into sessions: 
* A new session whenever the gap between two listens exceeds the minGab defined above ( currently 30 minutes).
* Filter: drop sessions shorter than minLen defined above (currently 5 listens).
* Truncate: sessions longer than maxLen defined above (currently 25 listens) keep only their first maxLen listens (the rest is dropped, not made into a new session).

In [ ]:
# read the listens of sample_2023_600users.csv, ordered by user, then timestamp
df = df.sort_values(["user_id", "listen_time_utc"], kind="stable").reset_index(drop=True)

# new session at each user's first listen, or when the gap to the previous listen exceeds MIN_GAP
gap = df.groupby("user_id")["listen_time_utc"].diff()
df["session_id"] = (gap.isna() | (gap > MIN_GAP)).cumsum()

# drop sessions shorter than MIN_LEN listens
session_size = df.groupby("session_id")["session_id"].transform("size")
sessions = df[session_size >= MIN_LEN].copy()

# give session new ids, and number / id the listens inside each session
sessions["session_id"] = sessions.groupby("session_id", sort=False).ngroup()
sessions["position_in_session"] = sessions.groupby("session_id").cumcount() + 1

# truncate long sessions: keep only the first MAX_LEN listens (the rest is dropped)
n_truncated = (sessions.groupby("session_id").size() > MAX_LEN).sum()
sessions = sessions[sessions["position_in_session"] <= MAX_LEN]

# write to the new sessions.csv: one row per listen, grouped by user and session
sessions.to_csv(SESSIONS_PATH, index=False)

# print the summery: 
n_all = df["session_id"].nunique()
n_kept = sessions["session_id"].nunique()
print(f"{n_all:,} sessions found, {n_kept:,} kept (>= {MIN_LEN} listens), "
      f"{n_truncated:,} truncated to the first {MAX_LEN} listens")

check = pd.read_csv(SESSIONS_PATH)
print(f"Total samples in {SESSIONS_PATH}: {check['session_id'].nunique():,} sessions, "
      f"{len(check):,} listens, {check['user_id'].nunique():,} users")

166,038 sessions found, 284 kept (>= 5 listens), 9 truncated to the first 25 listens
Total samples in sessions.csv: 284 sessions, 1,806 listens, 148 users


Next steps: 
* compute the features, 
* cluster, 
* train the model 
* evaluate the model against the baselines. 

**Problem with the current sample.** `sample_2023_600users.csv` kept each listen with only a 2.99% chance (see its README). That breaks real listening sessions apart: of 166,038 sessions found, only 284 have at least 5 listens (0.9% of all listens, 148 of 600 users). The sample is fine for trying out code, but not for RQ2.

**Options**

| Option | Assessment |
|---|---|
| A: the full 2023 extract (120 M listens, 11,529 users, 61 parquet files, ~11.6 GB) | Too large for pandas on a 30 GB laptop. Possible later if sessionizing is done in DuckDB, so that only `sessions.csv` has to fit in memory. |
| B: read the extract until we have ~1000 sessions | Biased: just the users and months that come first in the files. Also few sessions for clustering + training + evaluation. |
| **Proposed: a few users, all their listens** | Pick 50-100 users and keep **every** 2023 listen (no thinning). About 11k listens per user, so 100 users ≈ 1.1 M listens. That fits in memory and gives far more sessions than B (exact number unknown until the data exists). |

**How (on Hans's computer, where the parquet files are).** Use the existing chain in `Claude Reports/05_Test_Sample/Scripts`:
1. `make_sample_listens.py --extract "<project>/Extract/extract" --work <work folder> --users 100 --target 100000000`
   (a target above the users' total listens makes the keep fraction ≥ 1, so all listens are kept)
2. `make_keys.py`, then `link_musicbrainz.py`, then `build_sample_csv.py`. These add genre, tags, song length and the other MusicBrainz columns, which the raw extract doesn't have. Change the output file name so the current sample isn't overwritten.
3. Point `SAMPLE_PATH` below to the new csv.

**Variables**

*Script parameters (`make_sample_listens.py`)*

| Parameter | Proposed | Meaning |
|---|---|---|
| `--extract` | `<project>/Extract/extract` | folder with the `year=2023` parquet files |
| `--work` | any work folder | where the intermediate files go |
| `--users` | 100 (or 50) | number of users, picked by hash, so the same seed gives the same users |
| `--target` | 100000000 | target number of listens; set high so that nothing is thinned |
| `--seed` | 2023 | seed for picking users; keep it so the pick can be repeated |

**Open questions:** 50 or 100 users? Keep the same pool (users with 2,000-40,000 listens in 2023)? Do we want the real length of truncated sessions as a feature?